### 1. 패키지 설치

In [ ]:
%pip install -qU langgraph

### 2. 노드 반환값과 상태 갱신 규칙
- 노드 함수의 **반환값이 곧 상태 갱신 요청**이며 그래프가 현재 state에 병합함
- 반환값이 없으면(`None`) 갱신할 것이 없다는 뜻 → 현재 상태가 그대로 유지됨
- 받은 state를 그대로 반환해도 같은 값으로 덮어쓰는 것이라 결과는 동일함
- State에 정의되지 않은 필드를 반환하면 병합 대상이 아니므로 무시됨
- 이미 정의된 필드를 반환하면 그 값으로 갱신됨 (reducer가 없으면 덮어쓰기)
- 반환값은 전체 state가 아닌 **바뀐 필드만** 담은 부분 갱신(partial update)이어도 됨

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class State(TypedDict):
    address: str
    alive: bool
    age: int

def no_return(state: State):
    print('no_return 함수 실행')
    # 반환값 없음 → 갱신 요청이 없으므로 현재 상태가 그대로 유지됨

def return_state(state: State):
    print('return_state 함수 실행')
    return state                        # 받은 state를 그대로 반환 → 같은 값으로 덮어써서 변화 없음

def return_unknown(state: State):
    print('return_unknown 함수 실행')
    return {'job': 'developer'}         # State에 없는 필드 → 병합되지 않고 무시됨

def update_address(state: State):
    print('update_address 함수 실행')
    return {'address': '서울시 중랑구'}   # 정의된 필드 → 해당 값만 갱신 (나머지 필드는 유지)

graph = StateGraph(State)

graph.add_node('no_return', no_return)
graph.add_node('return_state', return_state)
graph.add_node('return_unknown', return_unknown)
graph.add_node('update_address', update_address)

graph.add_edge(START, 'no_return')
graph.add_edge('no_return', 'return_state')
graph.add_edge('return_state', 'return_unknown')
graph.add_edge('return_unknown', 'update_address')
graph.add_edge('update_address', END)

app = graph.compile()

app

### 3. 실행 (invoke)
- `no_return → return_state → return_unknown → update_address` 순서로 노드가 실행됨
- 앞의 세 노드는 상태를 바꾸지 못하고 `update_address`의 `address`만 반영됨
- 최종 결과에 `job` 키가 없는 것으로 State에 정의되지 않은 필드가 버려짐을 확인할 수 있음
- `alive`, `age`는 어떤 노드도 반환하지 않았으므로 입력값 그대로 남음

In [ ]:
app.invoke({'address': '서울시 강북구', 'alive': True, 'age': 30})

### 4. 정리
- 상태를 바꾸려면 **State에 정의된 필드**를 **반환**해야 함 (둘 중 하나라도 빠지면 변화 없음)
- 노드에서 state를 직접 수정하지 않고 바뀐 부분만 반환하는 방식이라 흐름 추적이 쉬움
- 새로운 정보를 담고 싶다면 노드가 아니라 State 정의부터 필드를 추가해야 함
- 같은 필드를 여러 노드가 반환하면 마지막 값이 남으므로 누적이 필요하면 reducer를 사용함